1. Load the Dataset 📂

Meaning: Tweets තියෙන dataset එක computer එකට load කරන එක.

Dataset එකේ:

text → User ලියපු tweet එක
airline_sentiment → ඒ tweet එක positive / negative / neutral ද කියන label එක

අපිට මේ columns 2 විතරයි ඕන.

In [ ]:
import pandas as pd

df = pd.read_csv("Tweets.csv")

df = df[["airline_sentiment", "text"]]

df.head()

,airline_sentiment,text
0,neutral,@VirginAmerica What @dhepburn said.
1,positive,@VirginAmerica plus you've added commercials t...
2,neutral,@VirginAmerica I didn't today... Must mean I n...
3,negative,@VirginAmerica it's really aggressive to blast...
4,negative,@VirginAmerica and it's a really big bad thing...


Text Preprocessing

2. Preprocess Text 🧹

Meaning: Tweets clean කරන එක.

Example:

"I LOVE this airline! Visit http://abc.com"

Clean කළාට පස්සේ වගේ වෙනවා:

"love airlin"

මේ step එකේ:

lowercase කරනවා → LOVE → love
URLs remove කරනවා
unnecessary stop words remove කරනවා → the, is, a වගේ
words වල root/stem එක ගන්නවා → loving → love වගේ

Purpose: Machine Learning model එකට text එක පහසුවෙන් process කරන්න.

In [ ]:
import nltk
import string
import re

from nltk.stem.porter import PorterStemmer

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords

ps = PorterStemmer()

def clean_text(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Tokenize
    text = nltk.word_tokenize(text)

    # Remove stopwords
    y = []
    for i in text:
        if i not in stopwords.words('english'):
            y.append(i)

    text = y[:]
    y.clear()

    # Stemming
    for i in text:
        y.append(ps.stem(i))

    return " ".join(y)

df["text_cleaned"] = df["text"].apply(clean_text)

df.head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,airline_sentiment,text,text_cleaned
0,neutral,@VirginAmerica What @dhepburn said.,@ virginamerica @ dhepburn said .
1,positive,@VirginAmerica plus you've added commercials t...,@ virginamerica plu 've ad commerci experi ......
2,neutral,@VirginAmerica I didn't today... Must mean I n...,@ virginamerica n't today ... must mean need t...
3,negative,@VirginAmerica it's really aggressive to blast...,@ virginamerica 's realli aggress blast obnoxi...
4,negative,@VirginAmerica and it's a really big bad thing...,@ virginamerica 's realli big bad thing


TF-IDF Feature Extraction

3. Feature Extraction – TF-IDF 🔢

Computer එකට "I love this airline" වගේ text එක directly understand කරන්න බැහැ.

ඒ නිසා text එක numbers බවට convert කරන්න ඕන.

TF-IDF කියන්නේ words වල importance එක calculate කරලා numbers වලට convert කරන method එක.

Example:

"great service"

↓

[0.52, 0.81, 0, 0, ...]

max_features=3000 කියන්නේ maximum 3000 important words/features use කරනවා කියන එක.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)

X = tfidf.fit_transform(df["text_cleaned"]).toarray()

Y = df["airline_sentiment"].values

print(X.shape)
print(Y.shape)

(14640, 3000)
(14640,)


4. Train/Test Split ✂️

Dataset එක කොටස් දෙකකට බෙදනවා:

80% → Training data
20% → Testing data

Training data → Model එකට ඉගෙන ගන්න.

Testing data → Model එක කොච්චර හොඳට ඉගෙනගෙන තියෙනවාද බලන්න.

Example:

10,000 tweets

→ 8,000 training
→ 2,000 testing

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=2
)

print(X_train.shape)
print(X_test.shape)

(11712, 3000)
(2928, 3000)


5. Train Naive Bayes Model 🤖

Naive Bayes machine learning algorithm එකට training tweets දෙනවා.

ඒක learn කරනවා:

"I love this airline" → Positive

"Worst airline ever" → Negative

"Flight was okay" → Neutral

ඊට පස්සේ unseen tweets වල sentiment predict කරනවා.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

model = MultinomialNB()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

nb_accuracy = accuracy_score(y_test, y_pred)

print("Naive Bayes Accuracy:", nb_accuracy)

Naive Bayes Accuracy: 0.7219945355191257


6. Train Random Forest Model 🌳

මේක තවත් Machine Learning algorithm එකක්.

Random Forest එකත් training data use කරලා sentiment predict කරන්න ඉගෙන ගන්නවා.

ඊට පස්සේ:

Naive Bayes Accuracy = ?
Random Forest Accuracy = ?

කියලා compare කරනවා.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred)

print("Random Forest Accuracy:", rf_accuracy)

Random Forest Accuracy: 0.75


7. Accuracy 🎯

Accuracy කියන්නේ model එක කොච්චර predictions හරියට කළාද කියන එක.

උදාහරණයක්:

Testing tweets = 2,000
Correct predictions = 1,600

Accuracy = 80%

ඒ කියන්නේ tweets 100කින් approximately 80ක් correctly classify කරනවා.

In [ ]:
print("Naive Bayes Accuracy:", nb_accuracy)
print("Random Forest Accuracy:", rf_accuracy)

Naive Bayes Accuracy: 0.7219945355191257
Random Forest Accuracy: 0.75
